In [ ]:
import sys
import os
import numpy as np
# Get the current directory of the notebook
notebook_dir = os.getcwd()

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
# Add the 2nd level parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(parent_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from inflow_model.blade_params import P600_Blade
from manager import FittingManager
from body_drag_objective import BodyDragObjective
from drone import parameters
import inflow_model.propeller_lookup_table as propeller_lookup_table

Prepare full-vehicle flight data for fitting. These should be the same datasets used for the full-vehicle BEMT fit.

In [ ]:
import data_factory

fitting_subfolder = "wind_free_space_cfd"
factory = data_factory.FittingFactory()
data_list = data_factory.generate_data_list(fitting_subfolder, '.csv')
print(f"Fitting Data list:")
for data in data_list:
    print(data)
datasets = factory.prepare_datasets(data_list)
datasets = datasets[:1] # select the 'good' data

Load the lookup table built from the single-rotor fit. This replaces live BET integration during the body-drag optimization — no aero coefficients are re-estimated here.

In [ ]:
lookup_table = propeller_lookup_table.PropellerLookupTable.Reader("p600_single_rotor")
fixed_aero_params = [19.788, 4.183, 0.011, np.radians(5.989)]  # cl_1, cl_2, cd, alpha_0
is_multiseed = True
init_guess = None  # or [0.5] to warm-start from a known value
manager = FittingManager.for_body_drag(
    P600_Blade(), parameters.P600(), datasets,
    lookup_table=lookup_table,
    fixed_aero_params=fixed_aero_params,
    init_guess=init_guess,
)

Start fitting. Thrust is computed from the lookup table; only the body drag coefficient `k_body_drag` is optimized against the vertical force residual.

In [ ]:
fitted_params = manager.run(is_multiseed=is_multiseed)
k_body_drag = fitted_params[0]
print(f"Fitted k_body_drag = {k_body_drag:.4f}")

Verify the fit by comparing the vertical-force loss with and without body drag.

In [ ]:
objective = BodyDragObjective(manager.model, lookup_table)
manager.model.adjust_resolution(is_fine_tune=True)

loss_no_drag = objective.get_loss([0.0], datasets)
loss_fitted  = objective.get_loss([k_body_drag], datasets)

print(f"Loss without body drag (k=0.0):       {loss_no_drag:.6f}")
print(f"Loss with fitted body drag (k={k_body_drag:.4f}): {loss_fitted:.6f}")

Plot the full-vehicle force fit with the fitted `k_body_drag` applied.

In [ ]:
fig0, fig1 = manager.plot(dataset_idx=2, lookup_table=lookup_table, is_using_lookup_table=True, sample_step=5)